# Trabalho de Séries Temporais — Grupo 5 (PLS)

O notebook tem 3 partes:

1. **Bases** — trata cada base e coloca na lista `bases` (é aqui que se adiciona base nova)
2. **Pipeline** — features, otimização e walk-forward. Já está pronto
3. **Comparação** — MAE, ranking, resíduos e importância das features

A parte 2 não compara nada: ela só produz as previsões fora da amostra de
5 bases × 4 modelos. A comparação é a parte 3.

In [ ]:
import os
# threads do BLAS em 1: quem paraleliza aqui e o joblib, nao a algebra linear
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import ast
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.ensemble import RandomForestRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

# 1. Bases

Cada base entra como um dicionário. O `df` precisa chegar aqui **já tratado**:
índice de data, frequência regular, sem furo no alvo.

As externas vão em duas listas, e isso muda como o pipeline as usa:

- `exog_conhecidas` — o valor futuro já é sabido na data da previsão
  (feriado, calendário, promoção planejada). Entra com o valor da própria data.
- `exog_defasadas` — não se sabe o valor futuro (temperatura observada,
  indicador divulgado com atraso). Entra só defasada em `h`.

In [ ]:
bases = []

def add_base(nome, df, alvo, m, h, exog_conhecidas=(), exog_defasadas=(),
             n_validacao=None, n_teste=None):
    base = {
        "nome": nome,
        "df": df.sort_index(),
        "alvo": alvo,
        "m": m,                                    # periodo sazonal
        "h": h,                                    # horizonte de previsao
        "exog_conhecidas": list(exog_conhecidas),
        "exog_defasadas": list(exog_defasadas),
        "n_validacao": n_validacao or 6 * h,       # onde escolhemos os hiperparametros
        "n_teste": n_teste or 6 * h,               # onde medimos o MAE final
    }
    bases.append(base)
    return base

Para escolher o `m` de uma base, a força da sazonalidade da STL — o mesmo
critério da Tarefa 03. Roda antes do `add_base`, só para decidir o número.

In [ ]:
def forca_sazonalidade(serie, m):
    res = STL(serie, period=m, robust=True).fit()
    return 1 - np.var(res.resid) / np.var(res.seasonal + res.resid)

def sugerir_m(serie, candidatos=range(2, 101), min_ciclos=3, top=10):
    forcas = []
    for m in candidatos:
        if len(serie) / m < min_ciclos:
            continue
        try:
            forcas.append((m, forca_sazonalidade(serie, m)))
        except Exception:
            continue

    forcas.sort(key=lambda x: -x[1])
    for m, f in forcas[:top]:
        print(f"m={m:>3} | forca={f:.4f}")
    return forcas[0][0]

## Base 1 — vendas semanais

Duas externas: `feriado` (conhecido de antemão) e `temperatura` (só se sabe
depois de acontecer, então entra defasada).

**Para adicionar base nova: copia esta célula, troca a leitura e o `add_base`.**
O resto do notebook roda sozinho em cima da lista `bases`.

In [ ]:
df = pd.read_excel("dados/dados_aula08_sarimax.xlsx")
df["data"] = pd.to_datetime(df["data"])
df = df.set_index("data").asfreq("W-MON")

add_base(
    nome="vendas_semanais",
    df=df,
    alvo="vendas",
    m=52,
    h=4,
    exog_conhecidas=["feriado"],
    exog_defasadas=["temperatura"],
)

df.head()

In [ ]:
pd.DataFrame([{
    "base": b["nome"],
    "obs": len(b["df"]),
    "inicio": b["df"].index.min().date(),
    "fim": b["df"].index.max().date(),
    "freq": b["df"].index.freqstr,
    "m": b["m"],
    "h": b["h"],
    "conhecidas": ", ".join(b["exog_conhecidas"]) or "-",
    "defasadas": ", ".join(b["exog_defasadas"]) or "-",
} for b in bases])

# 2. Pipeline

O que é comum a todos: features, exógenas, origens e o walk-forward.
Depois, um bloco por modelo — cada um com sua função de previsão e sua
função de otimização.

## 2.1 Features

Regra da seção inteira: uma linha com data `t` só pode usar informação de até
`t - h`. Por isso todo lag é `>= h` e todo `rolling` leva `shift(h)` antes.

A mesma tabela vai para o Random Forest e para o PLS.

In [ ]:
def criar_features(base, janelas=(4, 8, 12)):
    h, m = base["h"], base["m"]
    y = base["df"][base["alvo"]]

    tab = pd.DataFrame(index=base["df"].index)
    tab["y"] = y

    # lags: recentes (nivel atual) e sazonais (mesmo ponto do ciclo anterior)
    for lag in sorted({h, h + 1, h + 2, h + 3, m, m + h}):
        tab[f"lag_{lag}"] = y.shift(lag)

    # janelas moveis: o shift(h) antes do rolling e o que evita vazamento
    passado = y.shift(h)
    for j in janelas:
        tab[f"media_{j}"] = passado.rolling(j).mean()
        tab[f"desvio_{j}"] = passado.rolling(j).std()
    tab["delta_nivel"] = tab[f"media_{janelas[0]}"] - tab[f"media_{janelas[-1]}"]

    # calendario: vem do indice, entao e conhecido para qualquer data futura
    idx = base["df"].index
    tab["mes"] = idx.month
    tab["dia_semana"] = idx.dayofweek
    tab["semana"] = idx.isocalendar().week.astype(int).to_numpy()

    # encoding ciclico: dezembro e janeiro viram vizinhos, domingo e segunda tambem
    for col, periodo in [("mes", 12), ("dia_semana", 7), ("semana", 52)]:
        tab[f"{col}_sin"] = np.sin(2 * np.pi * tab[col] / periodo)
        tab[f"{col}_cos"] = np.cos(2 * np.pi * tab[col] / periodo)

    # externas
    for col in base["exog_conhecidas"]:
        tab[col] = base["df"][col]
    for col in base["exog_defasadas"]:
        tab[f"{col}_lag_{h}"] = base["df"][col].shift(h)

    # os lags criam NaN so no comeco da serie: essas linhas sao descartadas
    tab = tab.dropna()

    # coluna constante nao informa nada e quebra a padronizacao do PLS
    constantes = [c for c in tab.columns if c != "y" and tab[c].nunique() <= 1]
    return tab.drop(columns=constantes)

## 2.2 Exógenas do SARIMAX

Mesma regra de disponibilidade, no formato que o `statsmodels` consome.

In [ ]:
def criar_exog(base):
    if not base["exog_conhecidas"] and not base["exog_defasadas"]:
        return None

    exog = pd.DataFrame(index=base["df"].index)
    for col in base["exog_conhecidas"]:
        exog[col] = base["df"][col]
    for col in base["exog_defasadas"]:
        exog[f"{col}_lag_{base['h']}"] = base["df"][col].shift(base["h"])

    return exog.bfill()

## 2.3 Origens de previsão

A série é cortada em três partes:

```
[------- historico -------][-- validacao --][-- teste --]
```

A validação escolhe os hiperparâmetros. O teste só é tocado no final.
Os 4 modelos recebem exatamente as mesmas origens.

In [ ]:
def origens(base, etapa):
    idx = base["df"].index
    n, h = len(idx), base["h"]

    ini_teste = n - base["n_teste"]
    ini_val = ini_teste - base["n_validacao"]

    a, b = (ini_val, ini_teste) if etapa == "validacao" else (ini_teste, n)

    # passo = h -> blocos de previsao que nao se sobrepoem
    # a origem e a ultima data que o modelo enxerga
    return [idx[p] for p in range(a - 1, b - h, h)]

def datas_futuras(base, origem):
    idx = base["df"].index
    return idx[idx.get_loc(origem) + 1:][:base["h"]]

## 2.4 Walk-forward

Anda pelas origens, chama a função de previsão do modelo em cada uma e junta
tudo num DataFrame longo. É a única parte genérica: recebe `prever` como
argumento e não sabe qual modelo está rodando.

In [ ]:
def walk_forward(prever, params, base, tab, exog, lista_origens):
    linhas = []

    for origem in lista_origens:
        datas = datas_futuras(base, origem)
        pred = prever(params, base, tab, exog, origem)

        for passo, (data, valor) in enumerate(zip(datas, pred), start=1):
            linhas.append({
                "base": base["nome"],
                "origem": origem,
                "data": data,
                "passo": passo,
                "y_real": base["df"][base["alvo"]].loc[data],
                "y_previsto": float(valor),
            })

    saida = pd.DataFrame(linhas)
    saida["residuo"] = saida["y_real"] - saida["y_previsto"]
    return saida


def mae_validacao(prever, params, base, tab, exog):
    # roda o mesmo walk-forward na janela de validacao e devolve o MAE
    try:
        prev = walk_forward(prever, params, base, tab, exog, origens(base, "validacao"))
        return mean_absolute_error(prev["y_real"], prev["y_previsto"])
    except Exception:
        return np.inf

## 2.5 SARIMAX

As ordens não são fixas: `d` e `D` saem dos testes de estacionariedade e o
resto é grade `(p,d,q)×(P,D,Q,m)` selecionada por **BIC**, em paralelo com
`joblib` — mesmo procedimento da Tarefa 03.

A busca roda só uma vez por base, sobre o histórico até o início do teste, e
fica em cache. As ordens escolhidas ficam congeladas no walk-forward.

In [ ]:
def prever_sarimax(params, base, tab, exog, origem):
    y_treino = base["df"][base["alvo"]].loc[:origem]
    datas = datas_futuras(base, origem)

    ajuste = SARIMAX(
        y_treino,
        exog=None if exog is None else exog.loc[:origem],
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False, maxiter=100)

    previsao = ajuste.get_forecast(
        steps=base["h"], exog=None if exog is None else exog.loc[datas]
    )
    return previsao.predicted_mean.to_numpy()

In [ ]:
def eh_estacionaria(serie):
    # ADF: p < 0.05 -> estacionaria | KPSS: p > 0.05 -> estacionaria
    return adfuller(serie)[1] < 0.05 and kpss(serie)[1] > 0.05

def sugerir_d(serie, max_d=2):
    s = serie.dropna()
    for d in range(max_d + 1):
        if eh_estacionaria(s):
            return d
        s = s.diff().dropna()
    return max_d

def sugerir_D(serie, m, d):
    s = serie.diff(d).dropna() if d > 0 else serie
    s = s.diff(m).dropna()
    if len(s) < 2 * m:
        return 0
    return 1 if eh_estacionaria(s) else 0

In [ ]:
def _bic_sarimax(order, seasonal_order, y, exog):
    try:
        ajuste = SARIMAX(
            y, exog=exog, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False, maxiter=100)
        return ajuste.bic
    except Exception:
        return None


def otimizar_sarimax(base, tab, exog, usar_cache=True):
    # corte: nada do conjunto de teste entra na escolha das ordens
    corte = base["df"].index[-base["n_teste"] - 1]
    y = base["df"][base["alvo"]].loc[:corte]
    ex = None if exog is None else exog.loc[:corte]
    m = base["m"]

    d_sug = sugerir_d(y)
    D_sug = sugerir_D(y, m, d_sug)
    print(f"      d sugerido = {d_sug} | D sugerido = {D_sug} | m = {m}")

    combos = [
        ((p, d, q), (P, D, Q, m))
        for p in range(3) for d in range(d_sug + 1) for q in range(3)
        for P in range(3) for D in range(D_sug + 1) for Q in range(3)
    ]

    cache = f"resultados/cache_sarimax_{base['nome']}.csv"
    if usar_cache and os.path.exists(cache):
        busca = pd.read_csv(cache)
        busca["params"] = busca["params"].apply(ast.literal_eval)
    else:
        print(f"      testando {len(combos)} combinacoes em paralelo...")
        bics = Parallel(n_jobs=-1, backend="loky")(
            delayed(_bic_sarimax)(order, so, y, ex) for order, so in combos
        )
        busca = pd.DataFrame([
            {"params": {"order": order, "seasonal_order": so}, "criterio": "BIC", "valor": bic}
            for (order, so), bic in zip(combos, bics) if bic is not None
        ]).sort_values("valor").reset_index(drop=True)

        busca.assign(params=busca["params"].astype(str)).to_csv(cache, index=False)

    return busca.loc[0, "params"], busca

## 2.6 Holt-Winters

Referência univariada: não recebe exógenas. A grade cruza tendência,
sazonalidade e amortecimento, e a escolha é pelo MAE na validação.

In [ ]:
def prever_holtwinters(params, base, tab, exog, origem):
    y_treino = base["df"][base["alvo"]].loc[:origem]

    ajuste = ExponentialSmoothing(
        y_treino,
        trend=params["trend"],
        seasonal=params["seasonal"],
        damped_trend=params["damped"],
        seasonal_periods=base["m"] if params["seasonal"] else None,
        initialization_method="estimated",
    ).fit(optimized=True)

    return np.asarray(ajuste.forecast(base["h"]))


def otimizar_holtwinters(base, tab, exog):
    grade = [
        {"trend": t, "seasonal": s, "damped": d}
        for t in ("add", None)
        for s in ("add", "mul", None)
        for d in (True, False)
        if not (t is None and d)          # nao existe amortecimento sem tendencia
    ]

    # Holt-Winters nao usa exogenas: passa None
    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_holtwinters, p, base, tab, None) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 2.7 Random Forest

Usa a tabela de features. `n_jobs=1` no estimador de propósito: quem paraleliza
é a grade, e aninhar os dois deixa mais lento.

In [ ]:
def prever_rf(params, base, tab, exog, origem):
    treino = tab.loc[:origem]
    futuro = tab.loc[datas_futuras(base, origem)]

    reg = RandomForestRegressor(random_state=42, n_jobs=1, **params)
    reg.fit(treino.drop(columns="y"), treino["y"])

    return reg.predict(futuro.drop(columns="y"))


def otimizar_rf(base, tab, exog):
    grade = [
        {"n_estimators": n, "max_depth": d, "max_features": f,
         "min_samples_split": s, "min_samples_leaf": l}
        for n in (300, 600)
        for d in (None, 6, 12)
        for f in ("sqrt", 0.5)
        for s in (2, 5)
        for l in (1, 2)
    ]

    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_rf, p, base, tab, exog) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 2.8 PLS — modelo de especialização do grupo

Mesma tabela de features do Random Forest, para a comparação não ser decidida
por dados diferentes.

O `StandardScaler` não é opcional: o PLS constrói componentes maximizando
covariância com o alvo, então sem padronizar a feature de maior escala domina
tudo. `n_components` é o hiperparâmetro principal — poucos componentes
subajustam, muitos fazem o modelo voltar a ser uma regressão linear comum.

In [ ]:
def prever_pls(params, base, tab, exog, origem):
    treino = tab.loc[:origem]
    futuro = tab.loc[datas_futuras(base, origem)]

    reg = make_pipeline(StandardScaler(), PLSRegression(n_components=params["n_components"]))
    reg.fit(treino.drop(columns="y"), treino["y"])

    return np.asarray(reg.predict(futuro.drop(columns="y"))).ravel()


def otimizar_pls(base, tab, exog):
    # o limite e o numero de features disponiveis
    n_max = min(15, tab.shape[1] - 1)
    grade = [{"n_components": k} for k in range(1, n_max + 1)]

    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_pls, p, base, tab, exog) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 2.9 Rodar

Para cada base e cada modelo: otimiza fora do teste → congela os
hiperparâmetros → walk-forward no teste.

In [ ]:
MODELOS = [
    ("SARIMAX",       otimizar_sarimax,      prever_sarimax),
    ("Holt-Winters",  otimizar_holtwinters,  prever_holtwinters),
    ("Random Forest", otimizar_rf,           prever_rf),
    ("PLS",           otimizar_pls,          prever_pls),
]


def rodar(bases, modelos=MODELOS):
    previsoes, escolhidos, buscas = [], [], []

    for base in bases:
        tab = criar_features(base)
        exog = criar_exog(base)
        org = origens(base, "teste")

        print(f"[{base['nome']}] {len(base['df'])} obs | h={base['h']} | m={base['m']} | "
              f"{tab.shape[1] - 1} features | {len(org)} origens de teste")

        for nome, otimizar, prever in modelos:
            inicio = pd.Timestamp.now()
            print(f"   {nome}")

            params, busca = otimizar(base, tab, exog)

            # hiperparametros congelados: daqui pra frente ninguem mais mexe
            prev = walk_forward(prever, params, base, tab, exog, org)
            prev.insert(1, "modelo", nome)

            segundos = (pd.Timestamp.now() - inicio).total_seconds()
            print(f"      {params}  ({segundos:.0f}s)")

            previsoes.append(prev)
            busca["base"], busca["modelo"] = base["nome"], nome
            buscas.append(busca)
            escolhidos.append({
                "base": base["nome"],
                "modelo": nome,
                "params": str(params),
                "criterio": busca.loc[0, "criterio"],
                "valor": busca.loc[0, "valor"],
                "segundos": round(segundos, 1),
            })

    return (pd.concat(previsoes, ignore_index=True),
            pd.DataFrame(escolhidos),
            pd.concat(buscas, ignore_index=True))

In [ ]:
previsoes, parametros, buscas = rodar(bases)

previsoes.to_csv("resultados/previsoes.csv", index=False)
parametros.to_csv("resultados/hiperparametros.csv", index=False)
buscas.assign(params=buscas["params"].astype(str)).to_csv("resultados/busca.csv", index=False)

parametros

# 3. Comparação

Tudo daqui pra baixo sai de `previsoes` — que já são as previsões fora da
amostra, com os hiperparâmetros congelados.

A fazer: ranking dentro de cada base, vitórias e posição média, resíduos
(ACF + Ljung-Box) e importância das features (RF e PLS).

In [ ]:
mae = (previsoes
       .groupby(["base", "modelo"])
       .apply(lambda g: mean_absolute_error(g["y_real"], g["y_previsto"]))
       .rename("MAE")
       .reset_index()
       .sort_values(["base", "MAE"]))

mae